# Lecture 3 · Choosing and configuring language-model systems (Fable edition)

Companion notebook, generated from the code on the lecture pages (https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/). Run the cells in order; each one corresponds to a numbered step on the pages. Steps marked 💳 need `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `LLM_MODEL`) in the environment and are skipped otherwise.

## Step 1 — Load the course questions and compute the chance level

[1. Decide what “better” means for this assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/01-decide-what-better-means/)

In [ ]:
import json, math, pathlib, time, urllib.request
from collections import Counter

DATA = ("https://aiml-study-guides.onrender.com/datasets/generative-agentic-"
        "ai/l3-course-quiz.jsonl")
path = pathlib.Path("l3-course-quiz.jsonl")
if not path.exists():
    urllib.request.urlretrieve(DATA, path)
quiz = [json.loads(line) for line in path.open(encoding="utf-8")]
LETTERS = "ABCD"
chance = sum(1 / len(q["options"]) for q in quiz) / len(quiz)
print(f"{len(quiz)} questions; per lecture "
      f"{dict(sorted(Counter(q['lecture'] for q in quiz).items()))}; "
      f"options {dict(Counter(len(q['options']) for q in quiz))}; "
      f"chance {chance:.2f}")
print("example:", quiz[0]["question"][:110], "…")
for letter, option in zip(LETTERS, quiz[0]["options"]):
    print(f"  {letter}) {option[:100]}")

## Step 2 — Score the local models in two ways

[1. Decide what “better” means for this assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/01-decide-what-better-means/)

In [ ]:
import copy, warnings
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from torchao.quantization import quantize_, Int8WeightOnlyConfig
warnings.filterwarnings("ignore")

def as_prompt(q):
    """Take a quiz row; return its question and lettered options as text."""
    lines = [q["question"], ""] + [
        f"{l}) {o}" for l, o in zip(LETTERS, q["options"])
    ] + ["", "Answer with the letter only."]
    return "\n".join(lines)

def chat_ids(tok, text):
    """Wrap text as a user chat turn; return the prompt's token ids."""
    return tok.apply_chat_template([{"role": "user", "content": text}],
                                   add_generation_prompt=True,
                                   return_tensors="pt",
                                   return_dict=True)["input_ids"]

@torch.no_grad()
def by_letter(model, tok, q):
    """Method 1: show lettered options, read the scores of the letters as
    the reply's first token."""
    logits = model(chat_ids(tok, as_prompt(q))).logits[0, -1]
    letter_ids = [
        tok.encode(l, add_special_tokens=False)[0]
        for l in LETTERS[:len(q["options"])]
    ]
    return int(logits[letter_ids].argmax())

@torch.no_grad()
def by_option_text(model, tok, q):
    """Method 2: ask the question alone; score each option as the reply by
    its mean log-probability per token."""
    prompt = chat_ids(tok, q["question"])
    scores = []
    for option in q["options"]:
        ids = torch.cat([
            prompt,
            tok(option, return_tensors="pt",
                add_special_tokens=False)["input_ids"]
        ], dim=1)
        logp = model(ids).logits[0, :-1].log_softmax(-1)
        target = ids[0, 1:]
        scores.append(logp[prompt.shape[1] - 1:].gather(
            1, target[prompt.shape[1] - 1:, None]).mean().item())
    return max(range(len(scores)), key=scores.__getitem__)

def evaluate(name, model, tok, method):
    """Score every question with method; return accuracy, s/question, picks."""
    t, picks, right = time.perf_counter(), [], 0
    for k, q in enumerate(quiz, 1):
        picks.append(method(model, tok, q)); right += picks[-1] == q["answer"]
        if k % 20 == 0:
            print(
                f"    {name}, {method.__name__}: {k}/{len(quiz)}, "
                f"{right / k:.2f} so far", flush=True)
    return right / len(quiz), (time.perf_counter() - t) / len(quiz), Counter(
        LETTERS[p] for p in picks)

local = {}
for name in ("HuggingFaceTB/SmolLM2-135M-Instruct",
             "HuggingFaceTB/SmolLM2-360M-Instruct"):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name,
                                                 dtype=torch.float32).eval()
    local[name.split("/")[1]] = (model, tok)
# Lecture 2, Step 9: int8 weights
int8 = copy.deepcopy(local["SmolLM2-135M-Instruct"][0])
quantize_(
    int8, Int8WeightOnlyConfig(), filter_fn=lambda m, n: isinstance(
        m, torch.nn.Linear) and "lm_head" not in n)
local["SmolLM2-135M-Instruct, int8"] = (int8,
                                        local["SmolLM2-135M-Instruct"][1])

results = {}
for name, (model, tok) in local.items():
    for method in (by_letter, by_option_text):
        acc, sec, picked = evaluate(name, model, tok, method)
        results[name, method.__name__] = (acc, sec)
        print(
            f"{name:<28} {method.__name__:<15} accuracy {acc:.2f}  {sec:.2f} "
            f"s/question  picked {dict(sorted(picked.items()))}")

## Step 3 — Time the first token and the generation speed on the laptop

[1. Decide what “better” means for this assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/01-decide-what-better-means/)

In [ ]:
from transformers import TextIteratorStreamer
from threading import Thread

def timed_reply(model, tok, prompt_text, new_tokens=64):
    """Time to first token (prefill + first step) and decode speed of the
    following tokens."""
    ids = chat_ids(tok, prompt_text)
    streamer = TextIteratorStreamer(tok, skip_prompt=True)
    args = dict(input_ids=ids, max_new_tokens=new_tokens,
                min_new_tokens=new_tokens, do_sample=False,
                pad_token_id=tok.eos_token_id, streamer=streamer)
    t0 = time.perf_counter()
    Thread(target=model.generate, kwargs=args).start()
    first, pieces = None, 0
    for piece in streamer:
        if piece and first is None:
            first = time.perf_counter() - t0
        pieces += bool(piece)
    total = time.perf_counter() - t0
    return ids.shape[1], first, (new_tokens - 1) / (total - first)

# a long prompt made of real course text
notes_text = " ".join(q["question"] for q in quiz)
for name, (model, tok) in local.items():
    timed_reply(model, tok, "Hi")                                    # warm-up
    for label, text in (("short prompt", quiz[0]["question"]), ("long prompt",
                                                                notes_text)):
        n, ttft, tps = timed_reply(model, tok, text)
        print(
            f"{name:<22} {label:<12} {n:>5} prompt tokens: first token after "
            f"{ttft:5.2f} s, then {tps:5.1f} tokens/s")

## Step 4 — Ask two hosted models the same questions 💳 API

[1. Decide what “better” means for this assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/01-decide-what-better-means/)

In [ ]:
import os, re, statistics
from openai import OpenAI

# 💳 Same three variables as Lectures 6–8; without a key the hosted steps are
# skipped.
LLM_READY = bool(os.environ.get("LLM_API_KEY"))
client = OpenAI(api_key=os.environ["LLM_API_KEY"],
                base_url=os.environ.get("LLM_BASE_URL")) if LLM_READY else None
SMALL = os.environ.get("LLM_MODEL", "gpt-6-luna")
LARGE = os.environ.get("LLM_MODEL_LARGE", "gpt-6.1-sol")

def ask(model, prompt, effort="none", **extra):
    """One streamed request: reply text, seconds to the first text, seconds
    in total, token usage."""
    t0, first, parts, usage = time.perf_counter(), None, [], None
    for event in client.responses.create(model=model, input=prompt,
                                         reasoning={"effort": effort},
                                         stream=True, **extra):
        if event.type == "response.output_text.delta":
            first = first or time.perf_counter() - t0
            parts.append(event.delta)
        elif event.type == "response.completed":
            usage = event.response.usage
    return "".join(parts), first, time.perf_counter() - t0, usage

def letter(reply):
    """Take a reply; return the index of its first standalone A–D, or None."""
    m = re.search(r"\b([A-D])\b", reply)
    # None: the reply broke the format
    return LETTERS.index(m.group(1)) if m else None

def run_hosted(model, data, effort="none"):
    """Ask model each question in data; return a list of result dicts."""
    rows = []
    for k, q in enumerate(data, 1):
        try:
            reply, first, total, u = ask(model, as_prompt(q), effort)
            rows.append(
                dict(
                    pick=letter(reply), right=letter(reply) == q["answer"],
                    first=first, total=total, tin=u.input_tokens,
                    tout=u.output_tokens, reasoning=getattr(
                        u.output_tokens_details, "reasoning_tokens", 0) or 0))
        except Exception as e:  # keep going; report the error once
            rows.append(dict(error=f"{type(e).__name__}: {str(e)[:120]}"))
        if k % 20 == 0:
            ok = [r for r in rows if "error" not in r]
            print(
                f"    {model}, effort {effort}: {k}/{len(data)}, "
                f"{sum(r['right'] for r in ok) / max(len(ok), 1):.2f} so far",
                flush=True)
    return rows

def summary(rows):
    """Take run_hosted rows; return a line: accuracy, errors, time, tokens."""
    ok = [r for r in rows if "error" not in r]
    if not ok:
        return f"every request failed — {rows[0]['error']}"
    return (
        f"accuracy {sum(r['right'] for r in ok) / len(rows):.2f}; format "
        f"errors {sum(r['pick'] is None for r in ok)}; "
        f"errors {len(rows) - len(ok)}; first text after "
        f"{statistics.median(r['first'] or r['total'] for r in ok):.2f} s "
        f"(median); tokens per question "
        f"{statistics.mean(r['tin'] for r in ok):.0f} in, "
        f"{statistics.mean(r['tout'] for r in ok):.0f} out (of which "
        f"reasoning {statistics.mean(r['reasoning'] for r in ok):.0f})")

# the lowest effort each accepts: gpt-6.1-sol rejects "none" (HTTP 400)
EFFORT = {SMALL: "none", LARGE: "low"}
EXPLAIN = ("In about 150 words, explain what a KV cache stores and why it "
           "speeds up text generation.")

hosted = {}
if LLM_READY:
    for model in (SMALL, LARGE):
        hosted[model] = run_hosted(model, quiz, EFFORT[model])
        print(
            f"{model:<12} effort {EFFORT[model]:<4} {summary(hosted[model])}")
        speeds = []
        for _ in range(3):  # a longer reply, to measure generation speed
            reply, first, total, u = ask(model, EXPLAIN, EFFORT[model])
            visible = u.output_tokens - (getattr(u.output_tokens_details,
                                                 "reasoning_tokens", 0) or 0)
            speeds.append((first, visible / (total - first)))
        print(f"{'':<12} a {visible}-token explanation: first text after "
              f"{statistics.median(f for f, _ in speeds):.2f} s, "
              f"then {statistics.median(v for _, v in speeds):.0f} tokens/s "
              f"(median of 3)")
else:
    print("No LLM_API_KEY: hosted steps skipped (recorded outputs are "
          "on the page).")

## Step 5 — Compute the cost per correct answer (💳 API for the hosted rows)

[1. Decide what “better” means for this assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/01-decide-what-better-means/)

In [ ]:
# 💳 Prices per 1M tokens (input, output), standard tier, from the provider's
# pricing page, checked 2026-11-01.
PRICES = {"gpt-6-luna": (0.10, 0.50), "gpt-6.1-sol": (2.00, 10.00)}
# illustrative: a small rented cloud CPU; a laptop you own costs electricity
CPU_DOLLARS_PER_HOUR = 0.05

print(f"{'candidate':<30} {'accuracy':>8} {'$ per 1,000 questions':>22} "
      f"{'$ per 1,000 correct':>20}")
for name, method in (("SmolLM2-135M-Instruct", "by_letter"),
                     ("SmolLM2-360M-Instruct", "by_letter")):
    acc, sec = results[name, method]
    per_q = sec / 3600 * CPU_DOLLARS_PER_HOUR
    print(f"{name + ' (CPU)':<30} {acc:>8.2f} {1000 * per_q:>22.4f} "
          f"{1000 * per_q / acc:>20.4f}   (accuracy ≈ chance {chance:.2f})")
for model, rows in hosted.items():
    ok = [r for r in rows if "error" not in r]
    if not ok or model not in PRICES:
        continue
    p_in, p_out = PRICES[model]
    # reasoning tokens bill as output
    per_q = statistics.mean(r["tin"] * p_in + r["tout"] * p_out
                            for r in ok) / 1e6
    acc = sum(r["right"] for r in ok) / len(rows)
    print(f"{model:<30} {acc:>8.2f} {1000 * per_q:>22.4f} "
          f"{1000 * per_q / max(acc, 1e-9):>20.4f}")

## Step 6 — Count the notes and time prefill as the prompt grows

[2. Check what the model must fit into](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/02-check-what-the-model-must-fit-into/)

In [ ]:
CORPUS = ("https://aiml-study-guides.onrender.com/corpus/generative-agentic-"
          "ai/lectures-01-06.en.jsonl")
cpath = pathlib.Path("lectures-01-06.en.jsonl")
if not cpath.exists():
    urllib.request.urlretrieve(CORPUS, cpath)
notes = [json.loads(line)["text"] for line in cpath.open(encoding="utf-8")]
model, tok = local["SmolLM2-135M-Instruct"]
corpus_tokens = sum(len(tok(t)["input_ids"]) for t in notes)
print(f"course notes, Lectures 1–6: {len(notes)} sections, {corpus_tokens:,} "
      f"tokens (SmolLM2's tokenizer)")
for window in (8_192, 128_000, 1_050_000):
    print(
        f"  a {window:>9,}-token window holds "
        + ('all of it' if corpus_tokens < window
           else f'{window / corpus_tokens:.0%} of it')
    )

@torch.no_grad()
def prefill_seconds(n):
    """Run the model on the first n tokens of the notes; return the seconds."""
    ids = tok(" ".join(notes), return_tensors="pt", truncation=True,
              max_length=n)["input_ids"]
    t = time.perf_counter(); model(ids); return time.perf_counter() - t

prefill_seconds(256)                                               # warm-up
for n in (1024, 2048, 4096, 8192):
    s = prefill_seconds(n)
    print(
        f"  prefill of {n:>5,} tokens on this CPU: {s:5.2f} s "
        f"({1000 * s / n:4.2f} ms per token)", flush=True)

## Step 7 — Count the total and active parameters of two mixtures of experts

[3. Architectures and reasoning modes that change the trade-offs](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/03-architectures-and-reasoning-modes/)

In [ ]:
import json
from huggingface_hub import hf_hub_download

def moe_params(repo):
    """Take a repo id; return (experts, per token, total, active) counts."""
    c = json.load(open(hf_hub_download(repo, "config.json")))
    h, layers, vocab = c["hidden_size"], c["num_hidden_layers"], c[
        "vocab_size"]
    head = c.get("head_dim") or h // c["num_attention_heads"]
    # q, o and k, v
    attn = h * head * c["num_attention_heads"] * 2 + h * head * c[
        "num_key_value_heads"] * 2
    experts = c.get("num_local_experts") or c.get("num_experts")
    # gate, up, down
    expert = 3 * h * (c.get("moe_intermediate_size") or c["intermediate_size"])
    router = h * experts
    # input table and output layer
    embed = vocab * h * (1 if c.get("tie_word_embeddings") else 2)
    total = embed + layers * (attn + router + experts * expert)
    active = embed + layers * (attn + router +
                               c["num_experts_per_tok"] * expert)
    return experts, c["num_experts_per_tok"], total, active

for repo in ("mistralai/Mixtral-8x7B-v0.1", "Qwen/Qwen3-30B-A3B"):
    e, k, total, active = moe_params(repo)
    print(
        f"{repo:<28} {e:>3} experts, {k} per token: total {total / 1e9:5.1f} "
        f"B, active {active / 1e9:5.2f} B "
        f"({active / total:.0%});  memory for bf16 weights "
        f"{2 * total / 1e9:3.0f} GB;  compute per token ≈ 2 × active = "
        f"{2 * active / 1e9:.0f} GFLOPs")

## Step 8 — Answer the same questions at four reasoning efforts 💳 API

[3. Architectures and reasoning modes that change the trade-offs](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/03-architectures-and-reasoning-modes/)

In [ ]:
# 💳 The same questions at four reasoning efforts on the small hosted model.
if LLM_READY:
    p_in, p_out = PRICES.get(SMALL, (0, 0))
    for effort in ("none", "low", "medium", "high"):
        rows = hosted[SMALL] if effort == "none" else run_hosted(
            SMALL, quiz, effort)
        ok = [r for r in rows if "error" not in r]
        if not ok:
            print(
                f"effort {effort:<7} every request failed — {rows[0]['error']}"
            )
            continue
        cost = sum(r["tin"] * p_in + r["tout"] * p_out for r in ok) / 1e6
        print(f"effort {effort:<7} accuracy "
              f"{sum(r['right'] for r in ok) / len(rows):.2f}; reasoning "
              f"tokens per question "
              f"{statistics.mean(r['reasoning'] for r in ok):5.0f}; seconds "
              f"per question {statistics.mean(r['total'] for r in ok):5.2f}; "
              f"${cost:.4f} for {len(quiz)} questions")
else:
    print("No LLM_API_KEY: skipped.")

## Step 9 — Filter real next-token probabilities, then penalise a repetition loop

[4. Configure generation](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/04-configure-generation/)

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def next_token_probs(model, tok, text, temperature=1.0):
    """Take a model, tokenizer and prompt; return next-token probabilities."""
    logits = model(chat_ids(tok, text)).logits[0, -1]
    return F.softmax(logits / temperature, dim=-1)

def survivors(p, top_k=None, top_p=None, min_p=None):
    """Take probabilities and a filter; return (tokens kept, mass kept)."""
    keep = torch.ones_like(p, dtype=torch.bool)
    if top_k is not None:
        keep &= p >= p.topk(top_k).values[-1]
    if top_p is not None:
        sorted_p, idx = p.sort(descending=True)
        # keep tokens until the mass reaches top_p
        inside = sorted_p.cumsum(0) - sorted_p < top_p
        mask = torch.zeros_like(keep); mask[idx[inside]] = True; keep &= mask
    if min_p is not None:
        # at least min_p × the top token's probability
        keep &= p >= min_p * p.max()
    return int(keep.sum()), float(p[keep].sum())

model, tok = local["SmolLM2-135M-Instruct"]
for text in ("What does the KV cache store?",
             "Write a one-line slogan for a study assistant."):
    p = next_token_probs(model, tok, text)
    print(f"{text!r}: top token {tok.decode(int(p.argmax()))!r} with "
          f"{float(p.max()):.0%}")
    for label, kw in (("top-k 40", dict(top_k=40)),
                      ("top-p 0.9", dict(top_p=0.9)), ("min-p 0.1",
                                                       dict(min_p=0.1))):
        n, mass = survivors(p, **kw)
        print(f"   {label:<10} keeps {n:>5} tokens holding {mass:.0%} "
              f"of the probability")

# Penalties on a real repetition loop: Lecture 2's base model on "List three
# tips for learning a new language."
base_tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
base = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M",
                                            dtype=torch.float32).eval()
ids = base_tok("List three tips for learning a new language.",
               return_tensors="pt")["input_ids"]

def repeated_share(text, n=4):
    """Take a text; return the share of its word n-grams that repeat."""
    w = text.split()
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 1 - len(set(grams)) / max(len(grams), 1)

from transformers import LogitsProcessorList

class FrequencyPresencePenalty:
    """The hosted APIs' penalties: subtract frequency × (times a token
    already appeared) + presence × (appeared at all)."""
    def __init__(self, start, frequency=0.0, presence=0.0):
        """Store where the reply starts and the two penalty strengths."""
        self.start, self.frequency, self.presence = start, frequency, presence
    def __call__(self, input_ids, scores):
        """Take ids and scores; return the scores minus the two penalties."""
        counts = torch.zeros_like(scores).scatter_add_(
            1, input_ids[:, self.start:],
            torch.ones_like(input_ids[:, self.start:], dtype=scores.dtype))
        return scores - self.frequency * counts - self.presence * (counts > 0)

for label, kw in (
    ("greedy",
     {}), ("frequency_penalty 0.5",
           dict(logits_processor=LogitsProcessorList(
               [FrequencyPresencePenalty(ids.shape[1], frequency=0.5)]))),
    ("presence_penalty 0.5",
     dict(logits_processor=LogitsProcessorList(
         [FrequencyPresencePenalty(ids.shape[1], presence=0.5)]))),
    ("repetition_penalty 1.3", dict(repetition_penalty=1.3))):
    out = base.generate(ids, max_new_tokens=60, do_sample=False,
                        pad_token_id=base_tok.eos_token_id, **kw)
    text = base_tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    print(f"{label:<24} repeated 4-grams {repeated_share(text):4.0%} | "
          f"{text.strip().replace(chr(10), ' ⏎ ')[:110]}")

## Step 10 — Cut JSON with a stop string, then constrain the format (💳 API for the last lines)

[4. Configure generation](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/04-configure-generation/)

In [ ]:
# A stop sequence ends generation at a string; it knows nothing about the
# structure being written.
reply = ('{"lecture": 6, "reason": {"topic": "tools", "section": "6.2"}} '
         'Hope this helps!')
# what "stop at }" returns (the stop string is dropped)
cut = reply.split("}")[0]
for label, text in (("whole reply", reply), ('stopped at "}"', cut)):
    try:
        json.loads(text); verdict = "valid JSON"
    except json.JSONDecodeError as e:
        verdict = f"invalid JSON ({e.msg})"
    print(f"{label:<15} {verdict:<40} {text}")

# Constrained decoding: at each step only the tokens the format allows may be
# chosen.
model, tok = local["SmolLM2-135M-Instruct"]
ASK = ("Which lecture of the course (1–8) teaches this? Reply only with JSON "
       "like {{\"lecture\": 3}}.\n\n{q}")
digits = [tok.encode(str(d), add_special_tokens=False)[0] for d in range(1, 9)]

@torch.no_grad()
def free_reply(q):
    """Take a quiz question; return the model's free greedy reply."""
    ids = chat_ids(tok, ASK.format(q=q["question"]))
    out = model.generate(ids, max_new_tokens=12, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def constrained_reply(q):
    """Take a quiz question; return a reply forced into {"lecture": n}."""
    ids = torch.cat([
        chat_ids(tok, ASK.format(q=q["question"])),
        tok('{"lecture": ', return_tensors="pt",
            add_special_tokens=False)["input_ids"]
    ], dim=1)
    logits = model(ids).logits[0, -1]
    d = 1 + int(logits[digits].argmax())  # only "1"…"8" are allowed here
    return '{"lecture": ' + str(d) + "}"  # the closing brace is forced

for name, fn in (("free generation", free_reply), ("constrained",
                                                   constrained_reply)):
    valid = right = 0; chosen = Counter()
    for q in quiz:
        text = fn(q)
        try:
            obj = json.loads(text); valid += 1
            right += obj.get("lecture") == q["lecture"]
            chosen[obj.get("lecture")] += 1
        except (json.JSONDecodeError, AttributeError):
            pass
    print(
        f"{name:<16} valid JSON {valid}/{len(quiz)}, right lecture "
        f"{right}/{len(quiz)}, answers {dict(chosen)}   e.g. {fn(quiz[0])!r}")

# 💳 The provider's structured output: the reply must match a JSON schema.
if LLM_READY:
    SCHEMA = {
        "type": "json_schema", "name": "route", "strict": True, "schema": {
            "type": "object", "properties": {
                "lecture": {"type": "integer", "enum": list(range(1, 9))}
            }, "required": ["lecture"], "additionalProperties": False
        }
    }
    valid = right = 0; chosen = Counter()
    for q in quiz:
        try:
            reply, *_ = ask(SMALL, ASK.format(q=q["question"]),
                            text={"format": SCHEMA})
            obj = json.loads(reply); valid += 1
            right += obj["lecture"] == q["lecture"]
            chosen[obj["lecture"]] += 1
        except Exception as e:
            chosen[type(e).__name__] += 1
    print(f"{SMALL + ', schema':<16} valid JSON {valid}/{len(quiz)}, right "
          f"lecture {right}/{len(quiz)}, answers "
          f"{dict(sorted(chosen.items(), key=str))}")

## Step 11 — Repeat one request: greedy, sampled, seeded, hosted (💳 API for the last lines)

[4. Configure generation](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/04-configure-generation/)

In [ ]:
model, tok = local["SmolLM2-135M-Instruct"]
ids = chat_ids(tok, "Give a study tip in one sentence.")

def run(**kw):
    """Take generate() settings; return the model's reply to the prompt."""
    out = model.generate(ids, max_new_tokens=20, pad_token_id=tok.eos_token_id,
                         **kw)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

greedy = {run(do_sample=False) for _ in range(5)}
sampled = {run(do_sample=True, temperature=1.0) for _ in range(5)}
seeded = set()
for _ in range(5):
    torch.manual_seed(42); seeded.add(run(do_sample=True, temperature=1.0))
print(f"greedy, 5 runs:                  {len(greedy)} distinct reply")
print(f"sampling at T = 1, 5 runs:       {len(sampled)} distinct replies")
print(f"sampling with the seed reset:    {len(seeded)} distinct reply → "
      f"{next(iter(seeded))!r}")

# 💳 The hosted model: five identical requests at temperature 0 (allowed only
# without reasoning).
if LLM_READY:
    for effort in ("none", "low"):
        replies = []
        for _ in range(5):
            try:
                replies.append(
                    ask(SMALL, "Give a study tip in one sentence.", effort,
                        temperature=0)[0].strip())
            except Exception as e:
                replies.append(f"rejected — {str(e)[:90]}")
        print(f"{SMALL}, effort {effort}, temperature 0: {len(set(replies))} "
              f"distinct of 5 → {replies[0][:90]!r}")

## Step 12 — Probe both versions of the set with rules that ignore the question (💳 API for the hosted check)

[5. Read benchmark evidence](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-03-fable/05-read-benchmark-evidence/)

In [ ]:
BEFORE = ("https://aiml-study-guides.onrender.com/datasets/generative-agentic-"
          "ai/l3-course-quiz-before.jsonl")
bpath = pathlib.Path("l3-course-quiz-before.jsonl")
if not bpath.exists():
    urllib.request.urlretrieve(BEFORE, bpath)
# the same 67 questions as the site showed them before the fix
before = [json.loads(line) for line in bpath.open(encoding="utf-8")]

@torch.no_grad()
def options_only(model, tok, q):
    """Take model, tokenizer, question q; return its pick, question hidden."""
    return by_letter(model, tok, {**q, "question": "Which option is correct?"})

model, tok = local["SmolLM2-135M-Instruct"]
rules = {
    "always the second option": lambda q: 1,
    "always the longest option": lambda q: max(
        range(len(q["options"])), key=lambda i: len(q["options"][i])),
    "SmolLM2-135M, question hidden": lambda q: options_only(model, tok, q),
    "SmolLM2-135M, full question": lambda q: by_letter(model, tok, q),
}
print(f"{'rule (never reads the question unless stated)':<34} "
      f"{'as written before':>18} {'rebuilt set':>12}   chance {chance:.2f}")
for name, rule in rules.items():
    acc = [
        sum(rule(q) == q["answer"] for q in data) / len(data)
        for data in (before, quiz)
    ]
    print(f"{name:<34} {acc[0]:>18.2f} {acc[1]:>12.2f}", flush=True)

# 💳 Does the hosted model notice? The same 67 questions, as written before vs
# rebuilt.
if LLM_READY:
    for name, data in (("as written before", before), ("rebuilt set", quiz)):
        rows = hosted[SMALL] if data is quiz else run_hosted(SMALL, data)
        ok = [r for r in rows if "error" not in r]
        print(f"{SMALL}, {name:<18} accuracy "
              f"{sum(r['right'] for r in ok) / len(rows):.2f}")